<a href="https://colab.research.google.com/github/alishadaudi/eBookstore-client-project/blob/main/Powell's_eBookstore_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Code snippet 1
# Installing pycaret for ML modeling
!pip install git+https://github.com/pycaret/pycaret.git

In [ ]:
# Code snippet 2
# Importing classification components from pycaret
from pycaret.classification import *

In [ ]:
# Code snippet 3
# Importing the eBookstore Client dataset
import pandas as pd
data = pd.read_csv('https://docs.google.com/spreadsheets/d/1XSBE7VwTJ-xB300r2JAwZesaAbGTLvMgS4qRxXfDjys/pub?gid=0&single=true&output=csv')

# Display the data
data

,Customer ID,Title,First Name,Middle Name,Last Name,Suffix,Street Address1,Street Address2,City-ZipCode-State,Birth Date,...,Occupation,Gender,Marital Status,Home Owner Status,Number of Cars Owned,Number of Children At Home,Total Number of Children,Annual Income,Avg Monthly Spend,eBook Subscriber Flag
0,11000,NaN,Jon,V,Yang,NaN,3761 N. 14th St,NaN,Cleveland-44101-Ohio,4/8/66,...,Professional,M,M,1,0,0,2,137947,89,0
1,11001,NaN,Eugene,L,Huang,NaN,2243 W St.,NaN,Seattle-98101-Washington,5/14/65,...,Professional,M,S,0,1,3,3,101141,117,1
2,11002,NaN,Ruben,NaN,Torres,NaN,5844 Linden Land,NaN,Omaha-68101-Nebraska,8/12/65,...,Professional,M,M,1,1,3,3,91945,123,0
3,11003,NaN,Christy,NaN,Zhu,NaN,1825 Village Pl.,NaN,Fort Worth-76101-Texas,2/15/68,...,Professional,F,S,0,1,0,0,86688,50,0
4,11004,NaN,Elizabeth,NaN,Johnson,NaN,7553 Harness Circle,NaN,Oakland-94601-California,8/8/68,...,Professional,F,S,1,4,5,5,92771,95,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16514,13121,NaN,Latasha,E,Jimenez,NaN,7259 Birchwood,NaN,San Antonio-78201-Texas,3/22/65,...,Professional,F,M,1,4,5,5,101542,101,0
16515,26100,NaN,James,Garner,Ptaszynski,NaN,7345 Kenwal Rd.,NaN,Pittsburgh-15201-Pennsylvania,4/2/36,...,Professional,F,S,1,2,0,3,46549,46,0
16516,11328,NaN,Julian,M,Griffin,NaN,7398 Withersed Lane,NaN,Honolulu-96801-Hawaii,1/1/40,...,Management,M,M,1,2,0,5,133053,79,0
16517,23077,NaN,Marvin,A,Hernandez,NaN,1019 Book Road,NaN,Anaheim-92801-California,10/20/46,...,Skilled Manual,M,M,1,2,0,4,31930,65,0


In [ ]:
# Code snippet 4
import datetime
import pandas as pd

# Only process if 'Birth Date' column exists
if 'Birth Date' in data.columns:
    # Convert 'Birth Date' to datetime objects, explicitly specifying the format to handle two-digit years
    data['Birth Date'] = pd.to_datetime(data['Birth Date'], format='%m/%d/%y')

    # Correct years that are in the future (e.g., 2066 interpreted from '66' needs to be 1966)
    now = datetime.datetime.now()
    future_birth_year_mask = data['Birth Date'].dt.year > now.year
    data.loc[future_birth_year_mask, 'Birth Date'] -= pd.DateOffset(years=100)

    # Calculate age using the corrected birth dates
    data['Age'] = now.year - data['Birth Date'].dt.year

    # Adjust age if the birth month/day hasn't occurred yet this year
    data.loc[
        (now.month < data['Birth Date'].dt.month) |
        ((now.month == data['Birth Date'].dt.month) & (now.day < data['Birth Date'].dt.day)),
        'Age'
    ] -= 1

    # Drop the original 'Birth Date' column
    data = data.drop(columns=['Birth Date'])

# Display the updated DataFrame with the new 'Age' column (or current state if 'Birth Date' was already processed)
display(data.head())

,Customer ID,Title,First Name,Middle Name,Last Name,Suffix,Street Address1,Street Address2,City-ZipCode-State,Education Level,...,Gender,Marital Status,Home Owner Status,Number of Cars Owned,Number of Children At Home,Total Number of Children,Annual Income,Avg Monthly Spend,eBook Subscriber Flag,Age
0,11000,NaN,Jon,V,Yang,NaN,3761 N. 14th St,NaN,Cleveland-44101-Ohio,Bachelors,...,M,M,1,0,0,2,137947,89,0,60
1,11001,NaN,Eugene,L,Huang,NaN,2243 W St.,NaN,Seattle-98101-Washington,Bachelors,...,M,S,0,1,3,3,101141,117,1,60
2,11002,NaN,Ruben,NaN,Torres,NaN,5844 Linden Land,NaN,Omaha-68101-Nebraska,Bachelors,...,M,M,1,1,3,3,91945,123,0,60
3,11003,NaN,Christy,NaN,Zhu,NaN,1825 Village Pl.,NaN,Fort Worth-76101-Texas,Bachelors,...,F,S,0,1,0,0,86688,50,0,58
4,11004,NaN,Elizabeth,NaN,Johnson,NaN,7553 Harness Circle,NaN,Oakland-94601-California,Bachelors,...,F,S,1,4,5,5,92771,95,1,57


In [ ]:
# Code snippet 5

# Split the 'City-ZipCode-State' column into three new columns
data[['City', 'Zip Code', 'State']] = data['City-ZipCode-State'].str.split('-', expand=True)

# Drop the original 'City-ZipCode-State' column
data = data.drop(columns=['City-ZipCode-State'])

# Display the updated DataFrame to verify the changes
display(data.head())

,Customer ID,Title,First Name,Middle Name,Last Name,Suffix,Street Address1,Street Address2,Education Level,Occupation,...,Number of Cars Owned,Number of Children At Home,Total Number of Children,Annual Income,Avg Monthly Spend,eBook Subscriber Flag,Age,City,Zip Code,State
0,11000,NaN,Jon,V,Yang,NaN,3761 N. 14th St,NaN,Bachelors,Professional,...,0,0,2,137947,89,0,60,Cleveland,44101,Ohio
1,11001,NaN,Eugene,L,Huang,NaN,2243 W St.,NaN,Bachelors,Professional,...,1,3,3,101141,117,1,60,Seattle,98101,Washington
2,11002,NaN,Ruben,NaN,Torres,NaN,5844 Linden Land,NaN,Bachelors,Professional,...,1,3,3,91945,123,0,60,Omaha,68101,Nebraska
3,11003,NaN,Christy,NaN,Zhu,NaN,1825 Village Pl.,NaN,Bachelors,Professional,...,1,0,0,86688,50,0,58,Fort Worth,76101,Texas
4,11004,NaN,Elizabeth,NaN,Johnson,NaN,7553 Harness Circle,NaN,Bachelors,Professional,...,4,5,5,92771,95,1,57,Oakland,94601,California


In [ ]:
# Code snippet 6
print(data.shape)

(16519, 23)


In [ ]:
# Code snippet 7
# Displaying the dataset columns
data.columns

Index(['Customer ID', 'Title', 'First Name', 'Middle Name', 'Last Name',
       'Suffix', 'Street Address1', 'Street Address2', 'Education Level',
       'Occupation', 'Gender', 'Marital Status', 'Home Owner Status',
       'Number of Cars Owned', 'Number of Children At Home',
       'Total Number of Children', 'Annual Income', 'Avg Monthly Spend',
       'eBook Subscriber Flag', 'Age', 'City', 'Zip Code', 'State'],
      dtype='object')

In [ ]:
# Code snippet 8
# Displaying missing data percentage for every column
data.isnull().sum()/len(data)*100

,0
Customer ID,0.000000
Title,99.467280
First Name,0.000000
Middle Name,42.284642
Last Name,0.000000
Suffix,99.987893
Street Address1,0.000000
Street Address2,98.329197
Education Level,0.000000
Occupation,0.000000


In [ ]:
# Code snippet 9
# Setting the Target variable
y = 'eBook Subscriber Flag'

In [ ]:
# Code snippet 10
# Setting the ignored columns list
ignored_cols = ['Customer ID', 'Title', 'First Name', 'Middle Name', 'Last Name',
                'Suffix', 'Street Address1', 'Street Address2', 'Avg Monthly Spend']

In [ ]:
# Code snippet 11
# Setting the numerical variables
num_cols = ['Number of Cars Owned', 'Number of Children At Home',
       'Total Number of Children', 'Annual Income',
            'Age']

In [ ]:
# Code snippet 12
# Setting the categorical columns
cat_cols = ['Education Level','Occupation', 'Gender',
            'Marital Status', 'Home Owner Status',
            'City', 'Zip Code', 'State']

In [ ]:
# Code snippet 13
# Setting up the Pycaret classification setup
classification_setup = setup(data,
                             target = y,
                             ignore_features = ignored_cols,
                             numeric_features = num_cols,
                             categorical_features = cat_cols)

,Description,Value
0,Session id,1102
1,Target,eBook Subscriber Flag
2,Target type,Binary
3,Original data shape,"(16519, 23)"
4,Transformed data shape,"(16519, 22)"
5,Transformed train set shape,"(11563, 22)"
6,Transformed test set shape,"(4956, 22)"
7,Ignore features,9
8,Numeric features,5
9,Categorical features,8


In [ ]:
# Code snippet 14
# Invoking ML algorithms
compare_models()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.8009,0.8634,0.5966,0.7530,0.6656,0.5266,0.5339,1.7310
lightgbm,Light Gradient Boosting Machine,0.7978,0.8591,0.5992,0.7427,0.6632,0.5211,0.5273,1.0100
ada,Ada Boost Classifier,0.7902,0.8524,0.6226,0.7106,0.6635,0.5121,0.5146,0.7610
lr,Logistic Regression,0.7867,0.8350,0.5705,0.7290,0.6400,0.4916,0.4992,1.6890
lda,Linear Discriminant Analysis,0.7855,0.8350,0.5466,0.7401,0.6287,0.4827,0.4937,0.2250
rf,Random Forest Classifier,0.7853,0.8413,0.5932,0.7126,0.6473,0.4948,0.4992,1.6410
ridge,Ridge Classifier,0.7844,0.8351,0.5349,0.7445,0.6224,0.4772,0.4900,0.2180
xgboost,Extreme Gradient Boosting,0.7817,0.8433,0.6023,0.6992,0.6470,0.4903,0.4932,0.5340
et,Extra Trees Classifier,0.7680,0.8243,0.5895,0.6720,0.6280,0.4604,0.4626,1.4310
nb,Naive Bayes,0.7663,0.8054,0.4558,0.7422,0.5644,0.4170,0.4403,0.2160


Processing:   0%|          | 0/65 [00:00<?, ?it/s]

GradientBoostingClassifier(ccp_alpha=0.0, criterion='friedman_mse', init=None,
                           learning_rate=0.1, loss='log_loss', max_depth=3,
                           max_features=None, max_leaf_nodes=None,
                           min_impurity_decrease=0.0, min_samples_leaf=1,
                           min_samples_split=2, min_weight_fraction_leaf=0.0,
                           n_estimators=100, n_iter_no_change=None,
                           random_state=1102, subsample=1.0, tol=0.0001,
                           validation_fraction=0.1, verbose=0,
                           warm_start=False)

In [ ]:
# Code snippet 15
# Cross validation/building the best model with the best algo
best_model = create_model('gbc')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8073,0.8620,0.6042,0.7657,0.6754,0.5410,0.5487
1,0.8047,0.8579,0.5844,0.7732,0.6657,0.5314,0.5418
2,0.8064,0.8807,0.6078,0.7622,0.6763,0.5407,0.5477
3,0.7820,0.8524,0.5755,0.7129,0.6369,0.4837,0.4893
4,0.8097,0.8708,0.6276,0.7579,0.6866,0.5517,0.5567
5,0.8080,0.8691,0.6146,0.7613,0.6801,0.5451,0.5515
6,0.8106,0.8719,0.6224,0.7636,0.6858,0.5522,0.5581
7,0.8002,0.8668,0.5833,0.7593,0.6598,0.5218,0.5308
8,0.7993,0.8526,0.5938,0.7500,0.6628,0.5227,0.5299


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# Code snippet 16
# Evaluating the model (to get confusion matrix and feature importance)
evaluate_model(best_model)

interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

In [ ]:
# Code snippet 17
# Importing the client dataset
client_data = pd.read_csv('https://docs.google.com/spreadsheets/d/16s5TlD28DsgScenFfe7cMrzdCl75LQZA3mbOM_EekxA/pub?gid=1509803897&single=true&output=csv')

# Displaying client data
client_data

,CustomerID,Title,FirstName,MiddleName,LastName,Suffix,StreetAddress1,StreetAddress2,City-ZipCode-State,BirthDate,EducationLevel,Occupation,Gender,MaritalStatus,HomeOwnerStatus,NumCarsOwned,NumChildrenAtHome,TotalNumChildren,AnnualIncome
0,Customer_xx_1289,NaN,Trevor,NaN,Jenkins,NaN,4697 Yosemite Dr.,NaN,Seattle-98101-Washington,2/27/1953,Graduate Degree,Management,M,S,1,2,0,3,101674
1,Customer_xx_1290,NaN,Zachary,NaN,Anderson,NaN,2476 Mt. Whitney Way,NaN,Oklahoma City-73101-Oklahoma,5/14/1939,Bachelors,Management,M,M,1,2,0,5,131143
2,Customer_xx_1291,NaN,Frank,F,Navarro,NaN,Buergermeister-ulrich-str 2987,NaN,Tampa-33601-Florida,8/10/1961,Graduate Degree,Clerical,M,M,1,0,0,1,58637
3,Customer_xx_1292,NaN,Dennis,M,She,NaN,"828, rue de Berri",NaN,Charlotte-28201-North Carolina,7/8/1961,Graduate Degree,Clerical,M,S,1,0,0,1,45259
4,Customer_xx_1293,NaN,Randy,A,Zeng,NaN,2334 Brandywine Way,NaN,Plano-75023-Texas,8/21/1936,Graduate Degree,Clerical,M,M,1,0,0,1,46868
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2603,Customer_xx_3892,NaN,Cynthia,NaN,Kapoor,NaN,7045 Creekside Drive,NaN,Denver-80201-Colorado,7/28/1978,Partial College,Manual,F,S,0,1,0,0,20689
2604,Customer_xx_3893,NaN,Lance,NaN,Jimenez,NaN,Auf Der Steige 532,NaN,Miami-33101-Florida,11/26/1943,Bachelors,Clerical,M,M,1,0,0,1,59132
2605,Customer_xx_3894,NaN,Dominique,M,Saunders,NaN,3792 Westwood Ct.,NaN,Fresno-93701-California,4/26/1961,Graduate Degree,Clerical,F,M,0,0,0,3,65669
2606,Customer_xx_3895,NaN,Clayton,NaN,Zhang,NaN,"1080, quai de Grenelle",NaN,Corpus Christi-78401-Texas,3/5/1959,Bachelors,Clerical,M,M,1,0,0,3,59736


In [ ]:
# Code snippet 18
import datetime
import pandas as pd

# 1. Preprocess client_data to match the training data format

# Strip whitespace from all column names in client_data
client_data.columns = client_data.columns.str.strip()

# Calculate Age for client_data (similar to Code snippet 4)
if 'BirthDate' in client_data.columns:
    # Convert 'BirthDate' to datetime objects, assuming '%m/%d/%Y' format for client_data
    client_data['BirthDate'] = pd.to_datetime(client_data['BirthDate'], format='%m/%d/%Y', errors='coerce')

    now = datetime.datetime.now()
    client_data['Age'] = now.year - client_data['BirthDate'].dt.year

    # Adjust age if the birth month/day hasn't occurred yet this year
    client_data.loc[
        (now.month < client_data['BirthDate'].dt.month) |
        ((now.month == client_data['BirthDate'].dt.month) & (now.day < client_data['BirthDate'].dt.day)),
        'Age'
    ] -= 1

    # Drop the original 'BirthDate' column
    client_data = client_data.drop(columns=['BirthDate'])


# Split 'City-ZipCode-State' for client_data (similar to Code snippet 5)
if 'City-ZipCode-State' in client_data.columns:
    client_data[['City', 'Zip Code', 'State']] = client_data['City-ZipCode-State'].str.split('-', expand=True)
    client_data = client_data.drop(columns=['City-ZipCode-State'])

# Rename columns to match the training data (PyCaret setup)
client_data = client_data.rename(columns={
    'EducationLevel': 'Education Level',
    'MaritalStatus': 'Marital Status',
    'HomeOwnerStatus': 'Home Owner Status',
    'NumCarsOwned': 'Number of Cars Owned',
    'NumChildrenAtHome': 'Number of Children At Home',
    'TotalNumChildren': 'Total Number of Children',
    'AnnualIncome': 'Annual Income',
    'CustomerID': 'Customer ID', # Rename to match training data's ignored column
    'FirstName': 'First Name',   # Rename to match training data's ignored column
    'MiddleName': 'Middle Name', # Rename to match training data's ignored column
    'LastName': 'Last Name',
    'StreetAddress1': 'Street Address1', # Add this for consistency with training data
    'StreetAddress2': 'Street Address2'  # Add this for consistency with training data
})

# Add 'Avg Monthly Spend' column with a placeholder value (e.g., 0 or mean) if it doesn't exist
# This column was in the training data's ignored_cols but is not in client_data
if 'Avg Monthly Spend' not in client_data.columns:
    client_data['Avg Monthly Spend'] = 0 # Or replace with mean from training data if significant

# Create a list of all feature columns expected by the model (numeric + categorical)
feature_columns = num_cols + cat_cols

# Ensure client_data only contains these feature columns before prediction
# This step explicitly aligns the client_data columns with the model's expected features.
client_data = client_data[feature_columns]

# Predicting the eBook Subscribers status (eBook Subscriber Flag) for the client data set customers
predictions = predict_model(best_model, client_data)

# Exporting to a CSV file
predictions.to_csv('eBookstore_Client_Data_Predictions.csv')

# Display the predictions
predictions

,Number of Cars Owned,Number of Children At Home,Total Number of Children,Annual Income,Age,Education Level,Occupation,Gender,Marital Status,Home Owner Status,City,Zip Code,State,prediction_label,prediction_score
0,2,0,3,101674,73,Graduate Degree,Management,M,S,1,Seattle,98101,Washington,0,0.6831
1,2,0,5,131143,86,Bachelors,Management,M,M,1,Oklahoma City,73101,Oklahoma,0,0.9827
2,0,0,1,58637,64,Graduate Degree,Clerical,M,M,1,Tampa,33601,Florida,0,0.5807
3,0,0,1,45259,64,Graduate Degree,Clerical,M,S,1,Charlotte,28201,North Carolina,1,0.5261
4,0,0,1,46868,89,Graduate Degree,Clerical,M,M,1,Plano,75023,Texas,0,0.9825
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2603,1,0,0,20689,47,Partial College,Manual,F,S,0,Denver,80201,Colorado,0,0.8636
2604,0,0,1,59132,82,Bachelors,Clerical,M,M,1,Miami,33101,Florida,0,0.9726
2605,0,0,3,65669,64,Graduate Degree,Clerical,F,M,0,Fresno,93701,California,0,0.8831
2606,0,0,3,59736,67,Bachelors,Clerical,M,M,1,Corpus Christi,78401,Texas,0,0.8482
